# NB_SLV_POI_MEDIA_INFO

RECOMPUTE `lh_vv_silver.dbo.slv_poi_media_info`. Khóa `media_id`. Union Google và partner, một cover mỗi `poi_id`.


In [ ]:
p_exec_id = ""
p_run_id = ""
p_pl_name = ""
p_trg_fq = ""
p_pinned_json = ""
p_mode = "RUN"
p_force = "false"
p_history_limit = "1000"


In [ ]:
%run NB_LIB_TRANSFORM_SLV_GLD


In [ ]:
from pyspark.sql import Window
from pyspark.sql import types as T

GOOGLE, PARTNER = "google", "partner"
MEDIA_3P_COLS = ("media_dedup_key", "poi_id", "photo_api_uri", "display_order", "width_px", "height_px",
                 "photographer", "blob_url")
OUT_COLS = ("media_id", "poi_id", "source_name", "photo_api_uri", "display_order", "width_px", "height_px",
            "photographer", "blob_url", "thumbnail_url", "is_cover", "is_active")
MEDIA_SCHEMA = T.ArrayType(T.StructType([T.StructField("url", T.StringType()),
                                         T.StructField("order", T.IntegerType())]))


def _require(df, cols, name):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ConfigError(f"slv_poi_media_info: {name} thiếu {missing}")


def _live(df):
    return df.where(~F.coalesce(F.col("deleted"), F.lit(False)))


def media_state(raw):
    """(mảng, rỗng, không parse được). Rỗng: SQL null, trắng, JSON null, []. Chỉ parse chuỗi bắt đầu bằng '['."""
    blank = raw.isNull() | F.trim(raw).isin("", "null")
    parsed = F.when(F.trim(raw).startswith("["), F.from_json(raw, MEDIA_SCHEMA))
    empty = blank | (parsed.isNotNull() & (F.size(parsed) == 0))
    return parsed, empty, ~blank & parsed.isNull()


def google_media(media):
    return media.select(F.col("media_dedup_key").alias("media_id"), "poi_id", F.lit(GOOGLE).alias("source_name"),
                        "photo_api_uri", "display_order", "width_px", "height_px", "photographer", "blob_url",
                        F.lit(None).cast("int").alias("_pos"))


def partner_media(i18n, services, ctx=None):
    vi = (_live(i18n).where(F.col("locale") == "vi")
          .select(F.col("business_service_id").alias("_sid"),
                  uuid_md5(F.col("business_service_id"), F.lit(PARTNER)).alias("poi_id"), F.col("media").alias("_mi")))
    svc = _live(services).select(F.col("id").alias("_sid"), F.col("media").alias("_ms"))
    arr_i, empty_i, bad_i = media_state(F.col("_mi"))
    arr_s, _empty_s, bad_s = media_state(F.col("_ms"))
    docs = (vi.join(svc, "_sid", "left")
            .select("poi_id", F.when(empty_i, arr_s).otherwise(arr_i).alias("_arr"), empty_i.alias("_fallback"),
                    F.when(empty_i, bad_s).otherwise(bad_i).alias("_bad")))
    elems = docs.select("poi_id", F.posexplode("_arr").alias("_pos", "_e"))
    url = F.col("_e.url")
    kept = elems.where(url.isNotNull() & (F.trim(url) != ""))
    if ctx is not None:
        # Một lần persist, hai agg. Tránh năm action count() trên cùng lineage.
        cached = docs.persist()
        try:
            stats = cached.agg(
                F.sum(F.when(F.col("_bad"), 1).otherwise(0)).alias("unparsed"),
                F.sum(F.when(F.col("_fallback") & (F.size(F.col("_arr")) > 0), 1).otherwise(0)).alias("fallback"),
            ).first()
            ctx.warn("partner_media_unparsed", stats["unparsed"])
            ctx.metrics["partner_fallback_docs"] = int(stats["fallback"] or 0)
            n_drop = cached.select(F.posexplode("_arr").alias("_pos", "_e")).agg(
                F.sum(F.when(url.isNotNull() & (F.trim(url) != ""), 0).otherwise(1)).alias("n")).first()["n"]
            ctx.warn("partner_url_dropped", n_drop)
        finally:
            cached.unpersist()
    return kept.select(
        F.sha2(F.concat_ws("|", F.lit(PARTNER), F.col("poi_id"), F.trim(url)), 256).alias("media_id"), "poi_id",
        F.lit(PARTNER).alias("source_name"), F.lit(None).cast("string").alias("photo_api_uri"),
        F.col("_e.order").alias("display_order"), F.lit(None).cast("int").alias("width_px"),
        F.lit(None).cast("int").alias("height_px"), F.lit(None).cast("string").alias("photographer"),
        url.alias("blob_url"), "_pos")


def with_cover(df):
    # A8: mỗi poi_id một cover. display_order null thì dòng gặp đầu tiên theo A7.
    order = Window.partitionBy("poi_id").orderBy(F.col("display_order").asc_nulls_last(),
                                                 F.col("_pos").asc_nulls_last(), F.col("media_id").asc())
    return df.withColumn("is_cover", F.row_number().over(order) == 1)


def build(ctx):
    media, i18n, services = ctx.input("media_3p"), ctx.input("i18n"), ctx.input("services")
    _require(media, MEDIA_3P_COLS, "slv_3p_poi_media")
    _require(i18n, ("business_service_id", "locale", "media", "deleted"), "slv_pn_business_service_i18ns")
    _require(services, ("id", "media", "deleted"), "slv_pn_business_services")
    rows = with_cover(google_media(media).unionByName(partner_media(i18n, services, ctx)))
    rows = rows.withColumn("thumbnail_url", F.lit(None).cast("string")).withColumn("is_active", F.lit(True))
    # A8: row_number = 1 luôn là cover, nên không quét lại.
    ctx.warn("poi_without_cover", 0)
    return rows.select(*OUT_COLS)


SPEC = NodeSpec(
    trg_tbl="slv_poi_media_info",
    inputs={"media_3p": "slv_3p_poi_media", "i18n": "slv_pn_business_service_i18ns",
            "services": "slv_pn_business_services"},
    pk=("media_id",),
    build=build,
    required_inputs=("media_3p", "i18n"),
    unique_checks=(("one_cover_per_poi", ("poi_id",), "is_cover"),),
    block_checks=(("media_required_null",
                   "poi_id IS NULL OR source_name IS NULL OR is_cover IS NULL OR is_active IS NULL"),),
    soft_delete_set={"is_active": "false"},
    write_mode="MERGE",
)


In [ ]:
RES = run_node(SPEC, p_exec_id, p_run_id, p_pinned_json, p_mode, p_force, p_pl_name, p_trg_fq, p_history_limit)


In [ ]:
notebookutils.notebook.exit(finish_node(RES))
